# GPT Vision API 테스트 노트북
> FirstCare ML2: 식단 사진 분석 + 운동 캡처 인증

## 구조
1. 환경 설정 & API 키 로드
2. 유틸리티 함수
3. API 연결 테스트
4. 식단 사진 분석 — 무료 (REQ-HLTH-007)
5. 식단 사진 분석 — 유료 상세 리포트 (REQ-HLTH-007, -300pt)
6. 운동 캡처 OCR (REQ-CHAL-009)


## 1. 환경 설정

In [2]:
# # 필요한 패키지 설치 (최초 1회)
# !pip install openai python-dotenv

In [4]:
import os
import base64
import json
from openai import OpenAI
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API 키 로드 완료!")

API 키 로드 완료!


## 2. 유틸리티 함수

In [5]:
def encode_image(image_path: str) -> str:
    """이미지 파일을 base64로 변환"""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def get_image_media_type(image_path: str) -> str:
    """파일 확장자로 미디어 타입 결정"""
    ext = image_path.lower().split(".")[-1]
    media_types = {
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "png": "image/png",
        "gif": "image/gif",
        "webp": "image/webp"
    }
    return media_types.get(ext, "image/jpeg")

def call_vision_api(image_path: str, system_prompt: str, user_text: str, model: str = "gpt-4o-mini", detail: str = "low"):
    """Vision API 호출 공통 함수"""
    base64_image = encode_image(image_path)
    media_type = get_image_media_type(image_path)
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{base64_image}",
                            "detail": detail
                        }
                    },
                    {
                        "type": "text",
                        "text": user_text
                    }
                ]
            }
        ],
        max_tokens=1000
    )
    return response

def parse_and_print(response):
    """응답 파싱 및 출력"""
    raw = response.choices[0].message.content
    print("=== Raw 응답 ===")
    print(raw)
    print()
    
    # JSON 파싱
    try:
        result = json.loads(raw)
        print("=== 파싱 결과 ===")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result
    except json.JSONDecodeError:
        print("⚠️ JSON 파싱 실패 - prompt 수정 필요")
        return None

def print_usage(response, model="gpt-4o-mini"):
    """토큰 사용량 및 비용 출력"""
    usage = response.usage
    print(f"Input tokens:  {usage.prompt_tokens}")
    print(f"Output tokens: {usage.completion_tokens}")
    print(f"Total tokens:  {usage.total_tokens}")
    
    if model == "gpt-4o-mini":
        input_cost = usage.prompt_tokens / 1_000_000 * 0.15
        output_cost = usage.completion_tokens / 1_000_000 * 0.60
    else:  # gpt-4o
        input_cost = usage.prompt_tokens / 1_000_000 * 2.50
        output_cost = usage.completion_tokens / 1_000_000 * 10.00
    
    print(f"예상 비용: ${input_cost + output_cost:.6f}")

print("유틸리티 함수 로드 완료!")

유틸리티 함수 로드 완료!


## 3. API 연결 테스트

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "테스트입니다. '연결 성공'이라고만 답해주세요."}
    ],
    max_tokens=50
)

print(response.choices[0].message.content)

연결 성공


## 4. 식단 사진 분석 — 무료 (REQ-HLTH-007)

> 음식 사진 → 탄/단/지 비율 % + 한 줄 피드백
> 
> 모델: GPT-4o-mini | detail: low | 비용: ~$0.0002/장


In [65]:
# 식단 분석 — 무료 prompt
MEAL_FREE_PROMPT = """당신은 영양 상담사입니다.
사용자가 보낸 한 끼 식사 사진을 보고 영양 피드백을 제공합니다.
사용자의 심혈관 위험 요인이 함께 제공될 경우, 해당 정보를 반영하여 맞춤 피드백을 제공합니다.
 
아래 JSON 형식으로만 응답하세요. JSON 외의 텍스트는 절대 포함하지 마세요.
 
{
    "food_name": "대표 음식 이름",
    "food_items": ["개별 음식1", "개별 음식2"],
    "nutrition_ratio": {
        "carbohydrate_pct": 0,
        "protein_pct": 0,
        "fat_pct": 0
    },
    "sodium_level": "높음/보통/낮음",
    "feedback": "피드백 (2문장: 팩트 1문장 + 제안 1문장)"
}
 
피드백 작성 원칙:
- 첫 문장: 사실 기반 평가 ("~입니다", "~한 편입니다")
- 두 번째 문장: 개선 제안 ("~해보세요", "~을 곁들이면 좋습니다")
- 사용자 위험 요인이 제공된 경우, 해당 위험 요인에 맞춘 피드백 우선
  · 고혈압 위험군 → 나트륨 관련 피드백 우선
  · 고콜레스테롤 위험군 → 포화지방 관련 피드백 우선
  · 고혈당 위험군 → 당류/탄수화물 관련 피드백 우선
- 사용자 위험 요인이 없는 경우, 일반적인 영양 균형 기준으로 피드백
- "위험", "주의", "과다" 같은 경고 표현 금지
- "좋은 선택이에요", "훌륭해요" 같은 과한 칭찬 금지
- 사진에서 직접 확인할 수 있는 내용만 언급할 것
- 일반적인 조리법을 기반으로 추측하지 말 것
- 한 끼 식사 기준으로 평가
 
음식 인식 기준:
- 모든 종류의 음식을 인식할 것 (한식, 양식, 중식, 일식, 디저트, 간식 등)
- 한국 음식을 정확히 구분할 것:
  · 철판/불판 위 둥근 모양 내장류 → 곱창, 대창, 막창으로 인식
  · 얇게 썬 구운 고기 → 삼겹살, 목살, 갈비 등 부위 구분
  · 국/찌개/탕 종류 정확히 구분 (김치찌개, 된장찌개, 순두부 등)
  · 분식류 구분 (떡볶이, 순대, 튀김, 김밥 등)
- 여러 음식이 함께 있으면 food_items에 각각 나열
 
분석 기준:
- nutrition_ratio 세 값의 합은 100
- sodium_level 판단:
  · 높음: 국/찌개, 라면, 젓갈, 장류, 내장구이(양념)
  · 보통: 일반 반찬, 구이류
  · 낮음: 샐러드, 과일, 나물 위주
- 음식 사진이 아닌 경우: food_name을 "인식 불가"로 설정
- 한국어로 응답
"""


print("무료 식단 분석 prompt 설정 완료!")

무료 식단 분석 prompt 설정 완료!


In [66]:
# 식단 사진 분석 실행 — 무료
# ⬇️ 테스트할 음식 사진 경로
IMAGE_PATH = "test_images/마카롱.jpg"

# ⬇️ 테스트용 사용자 위험 요인 (실제로는 백엔드에서 넘겨줌)
risk_factors = "고혈압, 고콜레스테롤"

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=MEAL_FREE_PROMPT,
    user_text=f"이 음식 사진을 분석해주세요.\n\n사용자 위험 요인: {risk_factors}",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "food_name": "마카롱",
    "food_items": ["초콜릿 마카롱", "민트 마카롱", "라즈베리 마카롱", "바닐라 마카롱", "피스타치오 마카롱"],
    "nutrition_ratio": {
        "carbohydrate_pct": 70,
        "protein_pct": 10,
        "fat_pct": 20
    },
    "sodium_level": "낮음",
    "feedback": "당류가 높은 편입니다. 다른 건강한 간식으로 대체해보세요."
}

=== 파싱 결과 ===
{
  "food_name": "마카롱",
  "food_items": [
    "초콜릿 마카롱",
    "민트 마카롱",
    "라즈베리 마카롱",
    "바닐라 마카롱",
    "피스타치오 마카롱"
  ],
  "nutrition_ratio": {
    "carbohydrate_pct": 70,
    "protein_pct": 10,
    "fat_pct": 20
  },
  "sodium_level": "낮음",
  "feedback": "당류가 높은 편입니다. 다른 건강한 간식으로 대체해보세요."
}

Input tokens:  14906
Output tokens: 134
Total tokens:  15040
예상 비용: $0.002316


## 5. 식단 사진 분석 — 유료 상세 리포트 (REQ-HLTH-007, -300pt)

> 무료 분석 결과를 기반으로 상세 리포트 생성
> 
> 부족한 영양소를 채울 수 있는 음식 추천 포함


In [67]:
# 식단 분석 — 유료 상세 리포트 prompt
MEAL_PAID_PROMPT = """당신은 영양 상담사입니다.
사용자가 보낸 한 끼 식사 사진을 상세 분석하여 실천 가능한 개선 방안을 제공합니다.
사용자의 심혈관 위험 요인이 함께 제공될 경우, 해당 정보를 반영하여 맞춤 분석을 제공합니다.

아래 JSON 형식으로만 응답하세요. JSON 외의 텍스트는 절대 포함하지 마세요.

{
    "food_name": "대표 음식 이름",
    "food_items": ["개별 음식1", "개별 음식2"],
    "estimated_calories": 0,
    "nutrition_ratio": {
        "carbohydrate_pct": 0,
        "protein_pct": 0,
        "fat_pct": 0
    },
    "sodium_level": "높음/보통/낮음",
    "vitamin_info": {
        "level": "높음/보통/낮음",
        "detail": "주요 비타민 설명 (1문장)"
    },
    "mineral_info": {
        "level": "높음/보통/낮음",
        "detail": "주요 무기질 설명 (1문장)"
    },
    "detailed_analysis": {
        "strength": "이 식단의 장점 (1문장)",
        "improvement": "개선할 점 + 제안 (1~2문장)"
    },
    "recommendations": [
        {
            "nutrient": "보충하면 좋은 영양소",
            "foods": ["추천 음식1", "추천 음식2", "추천 음식3"],
            "reason": "사용자 건강 상태를 고려한 추천 이유 (1문장)"
        }
    ],
    "next_meal_suggestion": {
        "concept": "사용자 건강상태를 고려한 다음 끼니 컨셉",
        "menu_example": ["추천 메뉴1", "추천 메뉴2"],
        "reason": "이유 (1문장)"
    },
    "overall_score": 0,
    "feedback_summary": "종합 피드백 (2문장: 팩트 요약 + 실천 제안)"
}

피드백 작성 원칙:
- 탄/단/지 비율이 균형 잡힌 경우 (탄 40~60, 단 20~35, 지 15~30):
  사실 기반으로 인정하되 담백하게. 예: "탄수화물, 단백질, 지방의 비율이 균형 잡힌 식사입니다. 이 패턴을 유지해보세요."
- 그 외: 기존대로 팩트 1문장 + 제안 1문장
- 사용자 위험 요인이 제공된 경우, 해당 위험 요인에 맞춘 피드백 우선
  · 고혈압 위험군 → 나트륨 관련 피드백 우선
  · 고콜레스테롤 위험군 → 포화지방 관련 피드백 우선
  · 고혈당 위험군 → 당류/탄수화물 관련 피드백 우선
- 사용자 위험 요인이 제공된 경우, feedback_summary와 improvement에서 반드시 해당 위험 요인을 명시할 것
  · 예: "고혈압 관리를 위해 나트륨 섭취를 줄여보세요"
  · 예: "콜레스테롤 수치 관리를 위해 포화지방을 줄이는 것이 좋겠습니다"
- 사용자 위험 요인이 없는 경우, 일반적인 영양 균형 기준으로 피드백
- 좋은 점은 간단히 인정 (너무 과하게는 칭찬하지 말 것 -> 적당히 칭찬할 것)
- 개선점은 솔직하게 짚되, "~입니다" 팩트 전달 후 "~해보세요" 제안으로 마무리
- "위험", "주의", "과다" 같은 경고 표현 금지
- 사진에서 직접 확인할 수 있는 내용만 언급할 것
- 일반적인 조리법을 기반으로 추측하지 말 것
- 한 끼 식사 기준으로 평가

음식 인식 기준:
- 모든 종류의 음식을 인식할 것 (한식, 양식, 중식, 일식, 디저트, 간식 등)
- 한국 음식을 정확히 구분할 것:
  · 철판/불판 위 둥근 모양 내장류 → 곱창, 대창, 막창
  · 얇게 썬 구운 고기 → 삼겹살, 목살, 갈비 등 부위 구분
  · 국/찌개/탕, 분식류 정확히 구분
- 여러 음식이 함께 있으면 food_items에 각각 나열

점수 기준 (overall_score, 1~10):
- 8점 이상: 사용자의 노력을 인정하며 지속을 독려하는 톤
  예: "영양 밸런스가 잘 갖춰진 한 끼입니다. 이 패턴 유지하면 좋겠어요."
  recommendations는 "보충" 대신 "이 식단과 잘 어울리는 음식" 추천
- 5~7: 괜찮지만 보완 여지 있음
- 3~4: 한쪽으로 치우친 식사
- 1~2: 영양 균형이 많이 부족

분석 기준:
- vitamin_info: 사진 속 식재료 기반으로 비타민 수준 판단
  · 높음: 채소, 과일이 다양하게 포함된 식사
  · 보통: 일부 채소나 계란 등 포함
  · 낮음: 채소/과일이 거의 없는 식사
- mineral_info: 사진 속 식재료 기반으로 나트륨을 제외한 무기질(칼슘, 철분, 칼륨 등) 수준 판단
  · 높음: 해조류, 유제품, 견과류 포함
  · 보통: 고기, 계란 등 일부 포함
  · 낮음: 탄수화물 위주 식사
- nutrition_ratio 세 값의 합은 100
- estimated_calories: 사진 속 음식의 1인분 기준 추정 칼로리(kcal), 정수로 표시
- recommendations는 1개 작성
- 사용자 위험 요인이 제공된 경우, 해당 위험 요인에서 부족한 영양소 기반으로 추천
  · 예: 고혈압 → "칼륨 보충을 위해 바나나, 시금치를 추천합니다"
- 사용자 위험 요인이 없는 경우, 식단에서 부족한 영양소 기반으로 추천
- recommendations의 foods에는 food_items에 이미 포함된 음식/재료를 제외할 것
- next_meal_suggestion: 사용자 위험 요인이 있으면 해당 건강상태를 고려한 식단 추천
- next_meal_suggestion 메뉴는 한국에서 쉽게 먹을 수 있는 것
- 한국어로 응답
"""
print("유료 식단 분석 prompt 설정 완료!")

유료 식단 분석 prompt 설정 완료!


In [68]:
# 식단 사진 분석 실행 — 유료 상세 리포트
# ⬇️ 테스트할 음식 사진 경로 (무료와 같은 이미지로 비교)
IMAGE_PATH = "test_images/마카롱.jpg"

# ⬇️ 테스트용 사용자 위험 요인 (실제로는 백엔드에서 넘겨줌)
risk_factors = "고혈압, 고콜레스테롤" 

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=MEAL_PAID_PROMPT,
    user_text=f"이 음식 사진을 상세 분석해주세요.\n\n사용자 위험 요인: {risk_factors}",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "food_name": "마카롱",
    "food_items": ["마카롱"],
    "estimated_calories": 300,
    "nutrition_ratio": {
        "carbohydrate_pct": 70,
        "protein_pct": 10,
        "fat_pct": 20
    },
    "sodium_level": "높음",
    "vitamin_info": {
        "level": "낮음",
        "detail": "채소나 과일이 포함되지 않은 식사로 비타민이 부족할 수 있습니다."
    },
    "mineral_info": {
        "level": "낮음",
        "detail": "무기질이 거의 없는 고당분 음식입니다."
    },
    "detailed_analysis": {
        "strength": "달콤한 맛으로 식사 대용 간식으로 좋습니다.",
        "improvement": "고혈압과 고콜레스테롤 관리를 위해 당분과 나트륨 섭취를 줄이는 것이 좋습니다."
    },
    "recommendations": [
        {
            "nutrient": "비타민과 무기질 보충",
            "foods": ["과일", "채소", "견과류"],
            "reason": "영양 균형을 맞추기 위해 다양한 비타민과 무기질이 포함된 음식을 추천합니다."
        }
    ],
    "next_meal_suggestion": {
        "concept": "저염, 저당 식사",
        "menu_example": ["닭가슴살 샐러드", "채소 스튜"],
        "reason": "고혈압과 고콜레스테롤 관리를 위해 저염, 저당 수준의 음식을 섭취하는 것이 좋습니다."
    },
    "overall_score": 2,


## 6. 운동 캡처 OCR (REQ-CHAL-009)

> 운동 앱 스크린샷 → 운동 데이터 추출 + 챌린지 자동 완료
> 
> 모델: GPT-4o | detail: high | 비용: ~$0.0019/장


In [27]:
EXERCISE_PROMPT = """당신은 운동 기록 스크린샷을 분석하는 AI입니다.

사용자가 운동 앱 스크린샷을 보내면, 화면에 보이는 수치를 읽어서 아래 JSON 형식으로만 응답하세요.
JSON 외의 텍스트는 절대 포함하지 마세요.
반드시 순수 JSON만 출력하세요. ```json 같은 마크다운 코드블록으로 감싸지 마세요.

{
    "exercise_type": "운동 종류 (걷기/달리기/자전거/수영/등산/기타)",
    "metrics": {
        "steps": null,
        "distance_km": null,
        "duration_minutes": null,
        "calories_burned": null
    },
    "is_verified": true,
    "confidence": "high/medium/low",
    "note": ""
}

분석 규칙:
- 앱 종류에 관계없이, 화면에 보이는 숫자를 그대로 읽을 것
- 숫자 읽기 규칙:
  · 쉼표 구분: 11,166 → 11166
  · 시간 형식: 32:02 → 32분, 1:05:30 → 65.5분
  · 거리 단위: km과 m 구분 (m이면 km로 변환)
- 한국어/영어 UI 모두 인식:
  · "걸음", "steps" → steps
  · "분", "min" → duration_minutes
  · "칼로리", "kcal", "Cal" → calories_burned
  · "킬로미터", "km" → distance_km
- 해당 수치가 화면에 없으면 null 유지
- is_verified: 운동 기록 화면이 맞으면 true, 아니면 false
- confidence: 수치를 명확히 읽을 수 있으면 high, 일부 불확실하면 medium, 대부분 못 읽으면 low
- 운동 앱이 아닌 이미지: is_verified를 false, note에 "운동 기록 화면이 아닙니다" 기재
- 한국어로 응답
"""

In [29]:
# 운동 캡처 분석 실행
# ⬇️ 테스트할 운동 앱 스크린샷 경로
IMAGE_PATH = "test_images/운동기록.png"

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=EXERCISE_PROMPT,
    user_text="이 운동 앱 스크린샷을 분석해주세요.",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "exercise_type": "기타",
    "metrics": {
        "steps": null,
        "distance_km": null,
        "duration_minutes": 53,
        "calories_burned": 128
    },
    "is_verified": true,
    "confidence": "high",
    "note": ""
}

=== 파싱 결과 ===
{
  "exercise_type": "기타",
  "metrics": {
    "steps": null,
    "distance_km": null,
    "duration_minutes": 53,
    "calories_burned": 128
  },
  "is_verified": true,
  "confidence": "high",
  "note": ""
}

Input tokens:  25960
Output tokens: 70
Total tokens:  26030
예상 비용: $0.003936
